# 00 - Data cleaning

Checking the raw data for problems before doing anything else. The tables come from
postgres (`python src/data_load.py`). Names, emails and phones are dropped on load.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2:///credit_risk")

customers = pd.read_sql("SELECT * FROM customers", engine, parse_dates=["date_of_birth", "customer_since"])
customers = customers.drop(columns=["first_name", "last_name", "email", "phone"])
credit = pd.read_sql("SELECT * FROM credit_history", engine)
loans = pd.read_sql("SELECT * FROM loan_applications", engine, parse_dates=["application_date", "decision_date"])
txns = pd.read_sql("SELECT * FROM transactions", engine)

## Duplicates and missing values

In [2]:
for name, df in [("customers", customers), ("credit", credit), ("loans", loans), ("txns", txns)]:
    print(name, "duplicates:", df.duplicated().sum(), "| missing:", dict(df.isna().sum()[lambda s: s > 0]))

customers duplicates: 0 | missing: {}
credit duplicates: 0 | missing: {}
loans duplicates: 0 | missing: {'decision_date': np.int64(173), 'loan_default': np.int64(4097)}
txns duplicates: 0 | missing: {}


No duplicate rows. The only missing values are `decision_date` for Under Review apps and
`loan_default` for loans that weren't disbursed, which is expected.

## Category values

In [3]:
for col in ["gender", "marital_status", "education", "employment_type", "residence_type", "kyc_status"]:
    print(col, list(customers[col].unique()))
print("loan_type", list(loans["loan_type"].unique()))
print("channel", list(txns["channel"].unique()))

gender ['Male', 'Female']
marital_status ['Married', 'Single', 'Widowed', 'Divorced']
education ['Diploma', 'Graduate', 'Post Graduate', 'High School', 'Professional Degree']
employment_type ['Salaried', 'Unemployed', 'Retired', 'Self-Employed', 'Business Owner']
residence_type ['Family Owned', 'Mortgaged', 'Owned', 'Rented']
kyc_status ['Verified', 'Pending']
loan_type ['Business', 'Home', 'Auto', 'Gold', 'Personal', 'Education']
channel ['Net Banking', 'Debit Card', 'Credit Card', 'UPI', 'Branch', 'ATM']


In [4]:
pd.crosstab(txns["category"], txns["transaction_type"])

transaction_type,Credit,Debit
category,,
ATM Withdrawal,0,3240
Dining,0,3639
EMI,0,4770
Insurance,0,1160
Investment,0,1597
Refund,1180,0
Salary,5612,0
Shopping,0,6466
Transfer In,2443,0


All clean, no typos or odd spellings. Every category is always either credit or debit.

## Loan consistency checks

In [5]:
approved = loans["approval_status"] == "Approved"
print("approved amount > requested:", (loans["approved_amount"] > loans["requested_amount"]).sum())
print("not approved but has an amount:", (~approved & (loans["approved_amount"] > 0)).sum())
print("approved but not disbursed:", (approved & (loans["disbursed_flag"] == 0)).sum())
print("decision before application:", (loans["decision_date"] < loans["application_date"]).sum())

approved amount > requested: 0
not approved but has an amount: 0
approved but not disbursed: 0
decision before application: 0


## Debt to income is capped at 3

In [6]:
# recompute dti from the emi formula: (existing emi + new emi) / monthly income
r = loans["interest_rate"] / 1200
new_emi = loans["requested_amount"] * r / (1 - (1 + r) ** -loans["loan_term_months"])
dti = (loans["existing_emi"] + new_emi) / loans["applicant_monthly_income"]

capped = loans["debt_to_income_ratio"] == 3
print("capped rows:", capped.sum())
print("max difference on the other rows:", (dti - loans["debt_to_income_ratio"])[~capped].abs().max().round(3))
print("real dti on capped rows:", dti[capped].describe().round(1).to_dict())

capped rows: 1793
max difference on the other rows: 0.01
real dti on capped rows: {'count': 1793.0, 'mean': 8.2, 'std': 8.1, 'min': 3.0, '25%': 3.9, '50%': 5.5, '75%': 9.1, 'max': 86.0}


In [7]:
pd.crosstab(capped, loans['approval_status'])

approval_status,Approved,Rejected,Under Review
debt_to_income_ratio,,,
False,3257,2320,130
True,146,1604,43


The DTI matches the formula except for 1,793 rows where it's cut off at 3.0 (the real value
goes up to ~80). Most of these got rejected and only 146 were disbursed. Leaving the cap as it is,
it works like clipping outliers and anything above 3 is already very risky.

## Age is as of today, not the application date

In [8]:
m = loans.merge(customers, on="customer_id")
age_at_app = (m["application_date"] - m["date_of_birth"]).dt.days // 365
(m["age"] - age_at_app).value_counts().sort_index()

2    1894
3    3754
4    1852
Name: count, dtype: int64

`age` is 2-5 years higher than the age at application. In features I'm using age at the
application date instead.

## Dates

In [9]:
print("applications before customer_since:", (m["application_date"] < m["customer_since"]).sum())
too_young = customers["customer_since"] < customers["date_of_birth"] + pd.DateOffset(years=18)
print("customers who joined before 18:", too_young.sum())

applications before customer_since: 935
customers who joined before 18: 360


935 applications are dated before the customer's `customer_since`, which doesn't really make
sense. Tenure gets set to 0 for those. 360 customers joined before turning 18, maybe
minor/joint accounts, leaving them.

## Credit history

In [10]:
debt_to_limit = credit["total_outstanding_debt"] / credit["total_credit_limit"]
print("max diff vs utilization:", (debt_to_limit - credit["credit_utilization_ratio"]).abs().max())
print("outstanding > limit:", (credit["total_outstanding_debt"] > credit["total_credit_limit"]).sum())
print("90d lates > 30d lates:", (credit["num_late_payments_90d"] > credit["num_late_payments_30d"]).sum())

max diff vs utilization: 0.0020000000000000018
outstanding > limit: 0
90d lates > 30d lates: 0


Outstanding / limit is exactly the same as `credit_utilization_ratio`, so no point adding
it as a separate feature.

## Outliers

In [11]:
cols = {"annual_income": customers, "requested_amount": loans, "amount": txns}
pd.DataFrame({c: df[c].describe(percentiles=[.5, .99]) for c, df in cols.items()}).round(0)

,annual_income,requested_amount,amount
count,5000.0,7500.0,40079.0
mean,579576.0,2368134.0,14041.0
std,422597.0,2698808.0,18835.0
min,120000.0,31000.0,100.0
50%,474800.0,1288000.0,7928.0
99%,2099050.0,11517010.0,94926.0
max,4636900.0,12000000.0,359766.0


Skewed but nothing impossible (no negatives or zeros). Keeping them, tree models don't care
and for logistic regression the scaling is enough for now.

## Summary

The data is mostly clean so nothing gets changed in the database. The fixes go into `src/features.py`:
- age at application instead of `age`
- tenure clipped at 0
- no separate debt-to-limit feature (same as utilization)
- DTI cap at 3 kept as is